# NimbusPay ticket triage

Fine-tune Qwen2.5 1.5B Instruct with QLoRA so it maps a support ticket to the 8-field triage JSON. The training prompt is only the fixed system line from the data plus the ticket.

Run this on a fresh Colab **T4**. Runtime → Change runtime type → T4 GPU, then Run all. It should finish in under 3 hours. After it finishes, download `adapter/`, `predictions_test.jsonl`, `experiment_results.md`, and `error_examples.md`.

The submitted run is pre-registered: cleaned data, 4-bit QLoRA, rank 16, learning rate 2e-4, 2 epochs. Other runs change one thing at a time. Dev is only used for scoring.


In [ ]:
%%capture
!pip install -q unsloth


In [ ]:
import gc, json, os, shutil, subprocess, sys, time
from pathlib import Path

import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU. In Colab, switch the runtime to a T4.")

SEED = 3407
MAX_SEQ = 1024
LR = 2e-4
EPOCHS = 2
RANK = 16
MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct"
FINAL_NAME = "cleaned_qlora_r16"
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
FIXED_SYSTEM = "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."

torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))


## Data

`data/` must hold `train.jsonl`, `dev.jsonl`, and `test_inputs.jsonl`, or `training_assessment.zip` must sit next to this notebook. The zip is unpacked into `data/` and is not uploaded anywhere from here.


In [ ]:
def ensure_data():
    data = Path("data")
    data.mkdir(exist_ok=True)
    needed = ["train.jsonl", "dev.jsonl", "test_inputs.jsonl"]
    if all((data / name).exists() for name in needed):
        return data
    candidates = list(Path(".").glob("*.zip")) + list(Path("/content").glob("training_assessment.zip"))
    if not candidates:
        try:
            from google.colab import files
            print("Upload training_assessment.zip")
            files.upload()
            candidates = list(Path(".").glob("*.zip"))
        except ImportError:
            pass
    if not candidates:
        raise FileNotFoundError("Put the assignment jsonl files in data/ or upload the zip.")
    import zipfile
    with zipfile.ZipFile(candidates[0]) as zf:
        for info in zf.infolist():
            if Path(info.filename).name in needed and info.filename.endswith(".jsonl"):
                (data / Path(info.filename).name).write_bytes(zf.read(info))
    missing = [name for name in needed if not (data / name).exists()]
    if missing:
        raise FileNotFoundError(f"zip did not contain {missing}")
    return data

DATA = ensure_data()
assert Path("label_rules.py").exists() and Path("clean_train.py").exists() and Path("score.py").exists()

import clean_train
summary = clean_train.clean(
    DATA / "train.jsonl",
    DATA / "cleaned_train.jsonl",
    Path("cleaning_log.csv"),
    Path("cleaning_summary.json"),
)
raw_n = clean_train.write_raw_trainable(DATA / "train.jsonl", DATA / "raw_trainable.jsonl")
print(json.dumps(summary, indent=2))
print("raw trainable rows", raw_n)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_ids = {row["id"] for row in load_jsonl(DATA / "train.jsonl")}
dev_rows = load_jsonl(DATA / "dev.jsonl")
test_rows = load_jsonl(DATA / "test_inputs.jsonl")
assert train_ids.isdisjoint({row["id"] for row in dev_rows})
assert train_ids.isdisjoint({row["id"] for row in test_rows})
for row in dev_rows + test_rows:
    assert row["messages"][0]["content"] == FIXED_SYSTEM
print("dev", len(dev_rows), "test", len(test_rows))


## Why the sequence length is 1024

Most tickets are a few hundred characters. A few dozen are quoted email threads of several thousand characters, and the customer text is at the top of those threads. 1024 tokens covers the normal tickets with margin and keeps the complaint when a quote thread is cut off.


In [ ]:
lens = sorted(len(row["messages"][1]["content"]) for row in load_jsonl(DATA / "train.jsonl"))

def pct(p):
    return lens[min(len(lens) - 1, int(p * len(lens)))]

print(f"n={len(lens)} p50={pct(0.5)} p95={pct(0.95)} p99={pct(0.99)} max={lens[-1]}")
print("tickets longer than 1500 chars:", sum(n > 1500 for n in lens))


## Training helpers

Loss is applied only to the assistant turn. A check on the first batch stops the run if that mask is empty or if most of the ticket is being learned. Generation is greedy, `max_new_tokens=200`. The saved string is the decode with whitespace stripped, and nothing else.


In [ ]:
RULES_BASELINE = """Follow this labelling spec. Return only the JSON object.
category: fraud if the customer did not make or recognise the transaction (this wins); payment_failed if their payment, transfer, or top-up failed, was declined, or did not arrive; refund if money is owed back, including a double charge or a cancelled order; account_access if they cannot get in (PIN, passcode, password, OTP, locked account); kyc for identity checks; offers for cashback, coupons, referral, or reward points; otherwise other.
priority: P1 if category is fraud, or the amount is 50000 rupees or more, or they mention RBI, an ombudsman, consumer court, a lawyer, a legal notice, or legal action. Else P2 if account_access, or if payment_failed or refund and the amount is 5000 rupees or more. Else P3.
amount_paise: the transaction or claim amount in paise (rupees times 100). k means thousand, lakh means 100000. Ignore balance, limits, and fees. null if unstated.
txn_date: YYYY-MM-DD. Resolve yesterday, today, N days ago, kal, parso against Received. If the year is missing, use the latest such date on or before Received. Ignore dates on lines starting with >. null if unstated.
txn_id: NP plus 10 digits, uppercase, no spaces or hyphens. Ignore other reference numbers. null if absent.
channel: upi, card, netbanking, or wallet, only when stated. null otherwise.
For account_access, kyc, and other, amount_paise, txn_date, txn_id, and channel are null.
language: hi-en for Hinglish, else en.
needs_human: true if priority is P1 or they have already contacted support about this (third time, again and again, baar baar). An earlier ticket number alone does not count.
"""


def as_dataset(rows, tokenizer):
    texts = []
    for row in rows:
        text = tokenizer.apply_chat_template(
            row["messages"], tokenize=False, add_generation_prompt=False
        )
        texts.append({"text": text})
    return Dataset.from_list(texts)


def oversample_fraud(rows, times=3):
    extra = []
    for row in rows:
        label = json.loads(row["messages"][-1]["content"])
        copies = times if label.get("category") == "fraud" else 1
        extra.extend([row] * copies)
    return extra


def load_base(load_in_4bit, max_seq):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=max_seq,
        load_in_4bit=load_in_4bit,
        dtype=None,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def attach_lora(model, rank, max_seq):
    model = FastLanguageModel.get_peft_model(
        model,
        r=rank,
        target_modules=TARGET_MODULES,
        lora_alpha=rank,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        max_seq_length=max_seq,
    )
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable {trainable:,} / {total:,}")
    if trainable > 50_000_000:
        raise RuntimeError(f"trainable parameters {trainable} exceed 50M")
    return model, trainable


def generate(model, tokenizer, rows, system_override=None):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    out_rows = []
    for row in rows:
        messages = [dict(message) for message in row["messages"] if message["role"] != "assistant"]
        if system_override is not None:
            messages[0]["content"] = system_override
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            tokens = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        new_tokens = tokens[0, inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        out_rows.append({"id": row["id"], "output": text})
    tokenizer.padding_side = "right"
    return out_rows


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def score(gold_path, pred_path):
    proc = subprocess.run(
        [sys.executable, "score.py", "--gold", str(gold_path), "--pred", str(pred_path), "--json"],
        check=True, capture_output=True, text=True,
    )
    human = subprocess.run(
        [sys.executable, "score.py", "--gold", str(gold_path), "--pred", str(pred_path)],
        check=True, capture_output=True, text=True,
    )
    print(human.stdout)
    return json.loads(proc.stdout)


def free(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def train_and_eval(name, rows, load_in_4bit, rank, predict_test=False):
    print("\n====", name, "====")
    batch = 1 if not load_in_4bit else 2
    accum = 8 if batch == 1 else 4
    torch.cuda.reset_peak_memory_stats()
    model, tokenizer = load_base(load_in_4bit, MAX_SEQ)
    model, trainable = attach_lora(model, rank, MAX_SEQ)
    dataset = as_dataset(rows, tokenizer)
    sft_kwargs = dict(
        output_dir=f"outputs/{name}",
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=accum,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.03,
        dataset_text_field="text",
        packing=False,
        seed=SEED,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        save_strategy="no",
    )
    try:
        args = SFTConfig(**sft_kwargs, max_length=MAX_SEQ)
    except TypeError:
        args = SFTConfig(**sft_kwargs, max_seq_length=MAX_SEQ)
    try:
        trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=dataset, args=args)
    except TypeError:
        trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset, args=args)
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    batch0 = next(iter(trainer.get_train_dataloader()))
    labels = batch0["labels"]
    supervised = int((labels != -100).sum())
    fraction = supervised / labels.numel()
    print(f"assistant token fraction in first batch: {fraction:.1%}")
    if supervised == 0 or fraction > 0.65:
        raise RuntimeError("assistant-only loss mask looks wrong; refusing to train")
    started = time.time()
    trainer.train()
    minutes = (time.time() - started) / 60
    peak_gb = torch.cuda.max_memory_reserved() / (1024 ** 3)
    adapter_dir = Path("experiments") / name
    if adapter_dir.exists():
        shutil.rmtree(adapter_dir)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    dev_pred = Path("outputs") / f"{name}_dev.jsonl"
    dev_pred.parent.mkdir(parents=True, exist_ok=True)
    write_jsonl(dev_pred, generate(model, tokenizer, dev_rows))
    metrics = score(DATA / "dev.jsonl", dev_pred)
    record = {
        "name": name,
        "load_in_4bit": load_in_4bit,
        "rank": rank,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "train_rows": len(rows),
        "trainable": trainable,
        "minutes": round(minutes, 2),
        "peak_gpu_gb": round(peak_gb, 2),
        "exact_match": metrics["all"]["exact_match"],
        "mean_field_acc": metrics["all"]["mean_field_acc"],
        "json_valid": metrics["all"]["json_valid"],
        "fields": metrics["all"]["fields"],
    }
    if predict_test:
        write_jsonl("predictions_test.jsonl", generate(model, tokenizer, test_rows))
        final_dir = Path("adapter")
        if final_dir.exists():
            shutil.rmtree(final_dir)
        shutil.copytree(adapter_dir, final_dir)
        write_errors(dev_rows, load_jsonl(dev_pred), Path("error_examples.md"))
    free(trainer, model, tokenizer)
    return record


def write_errors(dev, preds, path, limit=10):
    by_id = {row["id"]: row["output"] for row in preds}
    blocks = ["# Dev failures", "", "Field-diff notes are a first pass over the mismatch, for the written error analysis.", ""]
    shown = 0
    for row in dev:
        gold = json.loads(row["messages"][-1]["content"])
        raw = by_id[row["id"]]
        try:
            pred = json.loads(raw.strip())
        except json.JSONDecodeError:
            pred = None
        exact = (
            isinstance(pred, dict)
            and set(pred) == set(KEYS)
            and all(pred.get(key) == gold[key] and type(pred.get(key)) is type(gold[key]) for key in KEYS)
        )
        if exact:
            continue
        shown += 1
        if shown > limit:
            break
        if not isinstance(pred, dict):
            diff = "output is not a JSON object"
            note = "Format failure: every field is scored wrong because the string is not one JSON object."
        else:
            bad = [key for key in KEYS if pred.get(key) != gold[key] or type(pred.get(key)) is not type(gold[key])]
            diff = ", ".join(bad) if bad else "extra or missing keys"
            notes = []
            if "txn_date" in bad:
                notes.append("Date: relative wording (kal, parso, N days ago) or a date on a line starting with `>`.")
            if "amount_paise" in bad:
                notes.append("Amount: balance, daily limit, k/lakh, or rupees left unconverted.")
            if "category" in bad:
                notes.append("Category: paraphrase the training set states in another way (unfamiliar payment, will not go through, charged twice).")
            if "needs_human" in bad or "priority" in bad:
                notes.append("Priority or needs_human: legal threat, the 50000/5000 thresholds, or a repeat-contact phrase.")
            if "txn_id" in bad or "channel" in bad:
                notes.append("Id or channel: spacing in NP ids, or a payment mode that was only implied.")
            note = " ".join(notes) or "The JSON parsed, but a field still differs."
        ticket = row["messages"][1]["content"][:1200]
        blocks.append(f"## {shown}. {row['id']}")
        blocks.append("")
        blocks.append("Ticket:")
        blocks.append("")
        blocks.append("```")
        blocks.append(ticket)
        blocks.append("```")
        blocks.append("")
        blocks.append(f"Gold: `{json.dumps(gold, ensure_ascii=False)}`")
        blocks.append("")
        blocks.append(f"Output: `{raw}`")
        blocks.append("")
        blocks.append(f"Fields: {diff}")
        blocks.append("")
        blocks.append(note)
        blocks.append("")
    if shown == 0:
        blocks.append("No dev failures. Exact match is 100% on this run.")
    path.write_text("\n".join(blocks), encoding="utf-8")
    print(f"wrote {path} with {min(shown, limit)} failures")


## Baselines

Same base model, no adapter. First with the fixed system prompt. Then with the schema added to that prompt. The schema prompt is not used for any training run.


In [ ]:
results = []

def remember(kind, metrics, minutes, peak):
    results.append({
        "name": kind,
        "load_in_4bit": True,
        "rank": 0,
        "epochs": 0,
        "learning_rate": 0,
        "train_rows": 0,
        "trainable": 0,
        "minutes": round(minutes, 2),
        "peak_gpu_gb": round(peak, 2),
        "exact_match": metrics["all"]["exact_match"],
        "mean_field_acc": metrics["all"]["mean_field_acc"],
        "json_valid": metrics["all"]["json_valid"],
        "fields": metrics["all"]["fields"],
    })

torch.cuda.reset_peak_memory_stats()
base, base_tok = load_base(True, MAX_SEQ)
started = time.time()
write_jsonl("outputs/baseline_fixed.jsonl", generate(base, base_tok, dev_rows))
remember("baseline_fixed_prompt", score(DATA / "dev.jsonl", "outputs/baseline_fixed.jsonl"), (time.time() - started) / 60, torch.cuda.max_memory_reserved() / 1024 ** 3)

torch.cuda.reset_peak_memory_stats()
started = time.time()
write_jsonl(
    "outputs/baseline_rules.jsonl",
    generate(base, base_tok, dev_rows, system_override=FIXED_SYSTEM + "\n\n" + RULES_BASELINE),
)
remember("baseline_rules_in_prompt", score(DATA / "dev.jsonl", "outputs/baseline_rules.jsonl"), (time.time() - started) / 60, torch.cuda.max_memory_reserved() / 1024 ** 3)
free(base, base_tok)


## Experiments and the submitted model

The cleaned rank-16 QLoRA run is the submitted model. It is trained once and reused as the control for the other four comparisons: raw labels, 16-bit LoRA, rank 8, and fraud rows repeated to 3×.


In [ ]:
cleaned_rows = load_jsonl(DATA / "cleaned_train.jsonl")
raw_rows = load_jsonl(DATA / "raw_trainable.jsonl")

runs = [
    (FINAL_NAME, cleaned_rows, True, RANK, True),
    ("raw_qlora_r16", raw_rows, True, RANK, False),
    ("cleaned_lora16_fp16", cleaned_rows, False, RANK, False),
    ("cleaned_qlora_r8", cleaned_rows, True, 8, False),
    ("cleaned_qlora_r16_fraud3x", oversample_fraud(cleaned_rows, 3), True, RANK, False),
]

for name, rows, four_bit, rank, is_final in runs:
    try:
        results.append(train_and_eval(name, rows, four_bit, rank, predict_test=is_final))
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print(name, "OOM")
        gc.collect()
        torch.cuda.empty_cache()
        results.append({
            "name": name, "exact_match": None, "mean_field_acc": None,
            "minutes": None, "peak_gpu_gb": None, "note": "out of memory",
        })
    table = ["| run | exact match | mean field acc | peak GPU GB | train minutes |", "|---|---:|---:|---:|---:|"]
    for row in results:
        table.append(
            f"| {row['name']} | {row.get('exact_match')} | {row.get('mean_field_acc')} | {row.get('peak_gpu_gb')} | {row.get('minutes')} |"
        )
    Path("experiment_results.md").write_text(
        "# Experiment results\n\nSubmitted model: `cleaned_qlora_r16`.\n\n" + "\n".join(table) + "\n",
        encoding="utf-8",
    )
    print("\n".join(table))


## What to keep

`adapter/` is the submitted PEFT adapter. `predictions_test.jsonl` is one line per test ticket. `experiment_results.md` fills the table in `report.md`. `error_examples.md` is the 10 dev failures for the error analysis. Copy the exact-match line for `cleaned_qlora_r16` into `README.md`.
